# Building `Legislators_data_processed`

Reproducible pipeline from the raw empirical edge list to the released dataset.

```
dataset_all_edges_coded_RD.csv            107,664 edges / 3,152 nodes
        |
        |  Step 1  party-blocked node reindexing (bijective relabeling)
        v
dataset_all_edges_coded_RD_reindexed.csv  107,664 edges  (graph unchanged)
        |
        |  Step 2  sink resolution: one outgoing edge per out-degree-0 node
        v
Legislators_data_processed.csv              107,785 edges
```

**Why Step 2 exists.** In the raw network 121 nodes have out-degree 0. In the
condensation (SCC-DAG) each is a terminal component, so out-Laplacian dynamics
`x' = -L_out x` has 121 independent invariant subspaces and global consensus is
structurally impossible regardless of the trust/distrust weights. Step 2 gives each
of those nodes exactly one outgoing edge into the dominant SCC, chosen conservatively
(same party, nearest belief, party-pair median weights) and fully logged in
`added_edges.csv`.

The pipeline is deterministic: no random seed is involved anywhere.

Requires `pandas`, `numpy`, `networkx`.

## Configuration

In [ ]:
from pathlib import Path
from typing import Dict, List, Set, Tuple

import networkx as nx
import numpy as np
import pandas as pd

DATA_DIR = Path(".")

RAW_CSV         = DATA_DIR / "dataset_all_edges_coded_RD.csv"
REINDEXED_CSV   = DATA_DIR / "dataset_all_edges_coded_RD_reindexed.csv"
REINDEX_MAP_CSV = DATA_DIR / "node_reindex_map_full.csv"
FINAL_CSV       = DATA_DIR / "Legislators_data_processed.csv"
ADDED_EDGES_CSV = DATA_DIR / "added_edges.csv"

# Party id blocks are assigned in this order: Democrats first, then Republicans.
PARTY_ORDER = ("Democratic", "Republican")

# Admissibility filter for edge weights:
#   trust - distrust >= TT      and      trust + distrust - 1 <= UT
TT, UT = 0.2, 0.8
EPS = 1e-6

EDGE_COLS = [
    "source", "target",
    "source_party", "target_party",
    "source_belief", "target_belief",
    "trust", "distrust",
]

## Helpers

`node_table` is the single source of truth for node attributes; it raises if any node
carries conflicting party/belief values across the edges it appears in, which is the
one silent-corruption failure mode this pipeline could otherwise hide.

In [ ]:
def norm_party(value) -> str:
    """Canonicalise party labels; raise on anything unexpected."""
    if pd.isna(value):
        return value
    s = str(value).strip().lower()
    if s.startswith("dem"):
        return "Democratic"
    if s.startswith("rep"):
        return "Republican"
    raise ValueError(f"Unrecognised party label: {value!r}")


def node_table(df: pd.DataFrame) -> pd.DataFrame:
    """One row per node: node, party, belief. Raises if attributes conflict."""
    src = df[["source", "source_party", "source_belief"]].rename(
        columns={"source": "node", "source_party": "party", "source_belief": "belief"})
    tgt = df[["target", "target_party", "target_belief"]].rename(
        columns={"target": "node", "target_party": "party", "target_belief": "belief"})
    nodes = pd.concat([src, tgt], ignore_index=True).drop_duplicates()
    if nodes["node"].duplicated().any():
        clashing = nodes.loc[nodes["node"].duplicated(keep=False), "node"].unique()
        raise ValueError(f"Nodes with inconsistent attributes: {clashing[:10]}")
    return nodes.sort_values("node").reset_index(drop=True)


def build_digraph(df: pd.DataFrame) -> nx.DiGraph:
    """Simple directed graph: self-loops dropped, parallel edges collapsed."""
    g = nx.DiGraph()
    g.add_nodes_from(pd.concat([df["source"], df["target"]]).unique().tolist())
    edges = df.loc[df["source"] != df["target"], ["source", "target"]].astype(int)
    g.add_edges_from(edges.itertuples(index=False, name=None))
    return g


def describe_structure(g: nx.DiGraph, label: str) -> dict:
    """Print and return the SCC statistics that Step 2 is judged on."""
    sccs = list(nx.strongly_connected_components(g))
    cond = nx.condensation(g, sccs)
    sinks = [n for n, deg in cond.out_degree() if deg == 0]
    info = {"label": label,
            "nodes": g.number_of_nodes(),
            "edges": g.number_of_edges(),
            "sccs": len(sccs),
            "sink_sccs": len(sinks),
            "largest_scc": max(len(s) for s in sccs)}
    print(f"[{label}] nodes={info['nodes']:,}  edges={info['edges']:,}  "
          f"SCCs={info['sccs']}  sink SCCs={info['sink_sccs']}  "
          f"largest SCC={info['largest_scc']:,}")
    return info

## Step 1 — Party-blocked node reindexing

Node ids are relabeled so each party occupies a contiguous block
(Democrats `0…1660`, Republicans `1661…3151`), ordered by ascending original id
within each block. This makes the adjacency and Laplacian matrices block-structured,
so within-party and cross-party blocks can be sliced without an index lookup.

This is a **pure relabeling**: no edge, node, or weight is added, removed, or changed.
Row order is canonicalised to `(source, target)`, which is a permutation of rows and
carries no graph meaning.

In [ ]:
def reindex_by_party(raw: pd.DataFrame,
                     party_order: Tuple[str, ...] = PARTY_ORDER
                     ) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Return (reindexed edge list, old -> new id map). Bijective relabeling."""
    nodes = node_table(raw)

    unknown = set(nodes["party"]) - set(party_order)
    if unknown:
        raise ValueError(f"Parties missing from PARTY_ORDER: {unknown}")

    blocks, next_id = [], 0
    for party in party_order:
        block = nodes.loc[nodes["party"] == party].sort_values("node").copy()
        block["new_id"] = range(next_id, next_id + len(block))
        next_id += len(block)
        blocks.append(block)

    mapping = (pd.concat(blocks, ignore_index=True)
               .sort_values("new_id")
               .reset_index(drop=True)[["node", "party", "belief", "new_id"]])
    old2new = dict(zip(mapping["node"], mapping["new_id"]))

    out = raw.copy()
    out["source"] = out["source"].map(old2new)
    out["target"] = out["target"].map(old2new)
    out = (out[EDGE_COLS]
           .sort_values(["source", "target"], kind="mergesort")
           .reset_index(drop=True))
    return out, mapping

In [ ]:
raw = pd.read_csv(RAW_CSV)
raw["source_party"] = raw["source_party"].map(norm_party)
raw["target_party"] = raw["target_party"].map(norm_party)

print(f"raw: {len(raw):,} edges, {len(node_table(raw)):,} nodes")
print(node_table(raw)["party"].value_counts().to_string())

reindexed, reindex_map = reindex_by_party(raw)
reindexed.to_csv(REINDEXED_CSV, index=False)
reindex_map.to_csv(REINDEX_MAP_CSV, index=False)

print(f"\nwrote {REINDEXED_CSV.name} and {REINDEX_MAP_CSV.name}")
reindexed.head()

In [ ]:
# The relabeling must not touch the graph: same edge multiset, same attributes.
_check = raw.copy()
_check["source"] = _check["source"].map(dict(zip(reindex_map["node"], reindex_map["new_id"])))
_check["target"] = _check["target"].map(dict(zip(reindex_map["node"], reindex_map["new_id"])))
_key = lambda d: (d[EDGE_COLS].round(9)
                  .sort_values(EDGE_COLS)
                  .reset_index(drop=True))
assert _key(_check).equals(_key(reindexed)), "Step 1 altered the graph"
print("Step 1 verified: bijective relabeling, graph and attributes preserved.")

## Step 2 — Sink resolution

For every sink SCC other than the anchor (the largest SCC), add **exactly one** edge
from a representative node into the anchor:

1. **Target** — the same-party anchor node with the closest belief; falls back to
   closest belief overall if no same-party candidate exists. Ties break on the lower
   node id, so the result is deterministic.
2. **Weights** — the median `(trust, distrust)` of the corresponding party-pair in the
   input data, so the added edges do not shift the weight marginals.
3. **Filter** — if those medians violate `(TT, UT)`, `nearest_feasible` applies the
   smallest symmetric correction that restores admissibility.
4. **Safeguards** — self-loops and duplicate edges are skipped by construction.

In [ ]:
def passes_filter(trust: float, distrust: float,
                  tt: float = TT, ut: float = UT, eps: float = 1e-9) -> bool:
    """trust - distrust >= tt, trust + distrust - 1 <= ut, both in [0, 1]."""
    if not (-eps <= trust <= 1 + eps) or not (-eps <= distrust <= 1 + eps):
        return False
    return (trust - distrust >= tt - eps) and (trust + distrust - 1 <= ut + eps)


def nearest_feasible(t0: float, d0: float,
                     tt: float = TT, ut: float = UT, eps: float = 1e-4
                     ) -> Tuple[float, float]:
    """Smallest symmetric correction of (t0, d0) into the admissible region."""
    t, d = float(t0), float(d0)
    if t - d < tt + eps:                       # raise trust, lower distrust
        gap = (tt + eps) - (t - d)
        t, d = min(1.0, t + gap / 2), max(0.0, d - gap / 2)
    if t + d - 1 > ut - eps:                   # lower both
        gap = (t + d - 1) - (ut - eps)
        t, d = max(0.0, t - gap / 2), max(0.0, d - gap / 2)
    t, d = float(np.clip(t, 0.0, 1.0)), float(np.clip(d, 0.0, 1.0))
    if not passes_filter(t, d, tt, ut):
        t, d = 0.60, 0.35                      # documented safe fallback
    return t, d


def party_pair_medians(df: pd.DataFrame) -> Dict[Tuple[str, str], Tuple[float, float]]:
    """Median (trust, distrust) for each (source_party -> target_party) pair."""
    return {key: (float(sub["trust"].median()), float(sub["distrust"].median()))
            for key, sub in df.groupby(["source_party", "target_party"])}

In [ ]:
def resolve_sinks(df: pd.DataFrame, tt: float = TT, ut: float = UT
                  ) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Give every sink SCC except the anchor one outgoing edge into the anchor.

    Returns (augmented edge list, added-edge log with per-edge diagnostics).
    """
    graph = build_digraph(df)
    sccs: List[Set[int]] = list(nx.strongly_connected_components(graph))
    cond = nx.condensation(graph, sccs)
    node_to_scc: Dict[int, int] = cond.graph["mapping"]

    sink_ids = [n for n, deg in cond.out_degree() if deg == 0]
    sizes = {i: len(s) for i, s in enumerate(sccs)}
    anchor_id = max(sizes, key=sizes.get)                    # largest SCC
    anchor_nodes = [v for v, cid in node_to_scc.items() if cid == anchor_id]

    nodes = node_table(df).set_index("node")
    party, belief = nodes["party"].to_dict(), nodes["belief"].to_dict()
    medians = party_pair_medians(df)
    existing = set(map(tuple, df[["source", "target"]].astype(int).to_numpy()))

    records = []
    for scc_id in sorted(sink_ids):
        if scc_id == anchor_id:
            continue                                         # anchor stays the sink
        members = sorted(v for v, cid in node_to_scc.items() if cid == scc_id)
        u = int(members[0])                                   # representative
        sp, ub = party[u], belief[u]

        same_party = [w for w in anchor_nodes if party.get(w) == sp]
        candidates = same_party if same_party else anchor_nodes
        rule = "same_party_closest_belief" if same_party else "closest_belief"
        v = int(min(candidates, key=lambda w: (abs(belief[w] - ub), w)))
        if u == v or (u, v) in existing:
            continue

        tp = party[v]
        t0, d0 = medians.get((sp, tp), (0.60, 0.35))
        ok = passes_filter(t0, d0, tt, ut)
        t, d = (float(t0), float(d0)) if ok else nearest_feasible(t0, d0, tt, ut)

        records.append({"source": u, "target": v,
                        "source_party": sp, "target_party": tp,
                        "source_belief": belief[u], "target_belief": belief[v],
                        "median_trust": float(t0), "median_distrust": float(d0),
                        "median_passed_filter": bool(ok),
                        "final_trust": float(t), "final_distrust": float(d),
                        "same_party_found": bool(same_party),
                        "selection_rule": rule})

    added = pd.DataFrame(records)
    if added.empty:
        return df[EDGE_COLS].copy(), added

    new_edges = added.rename(columns={"final_trust": "trust",
                                      "final_distrust": "distrust"})[EDGE_COLS]
    augmented = pd.concat([df[EDGE_COLS], new_edges], ignore_index=True)
    return augmented, added

In [ ]:
before = describe_structure(build_digraph(reindexed), "before")

final, added = resolve_sinks(reindexed)

after = describe_structure(build_digraph(final), "after")
print(f"\nadded {len(added)} edges "
      f"({len(added) / len(final) * 100:.3f}% of the final edge list); "
      f"sink SCCs {before['sink_sccs']} -> {after['sink_sccs']}")

final.to_csv(FINAL_CSV, index=False)
added.to_csv(ADDED_EDGES_CSV, index=False)
print(f"wrote {FINAL_CSV.name} and {ADDED_EDGES_CSV.name}")

In [ ]:
# Diagnostics of the intervention: how often each rule fired, and its footprint.
print("selection rule:\n", added["selection_rule"].value_counts().to_string())
print("\nmedians passed filter:\n", added["median_passed_filter"].value_counts().to_string())
print("\ndirection of added edges:\n",
      added.groupby(["source_party", "target_party"]).size().to_string())
print(f"\nmax |belief(source) - belief(target)| among added edges: "
      f"{(added['source_belief'] - added['target_belief']).abs().max():.4f}")
print("\nweights used:\n",
      added[["final_trust", "final_distrust"]].drop_duplicates().to_string(index=False))
added.head()

## Validation

These are the invariants the released dataset is expected to satisfy. They run in a
second or two and are worth keeping in the notebook: if an upstream file is ever
replaced, this cell is what catches it.

In [ ]:
n_nodes = len(node_table(final))

assert len(final) == len(reindexed) + len(added)
assert (final["source"] != final["target"]).all(),            "self-loop present"
assert not final.duplicated(["source", "target"]).any(),      "duplicate edge present"
assert final.notna().all().all(),                             "missing values present"
assert final[["trust", "distrust"]].apply(lambda c: c.between(0, 1)).all().all(), \
    "weights outside [0, 1]"
assert after["sink_sccs"] == 1,                               "more than one sink SCC"
assert final.iloc[:len(reindexed)].reset_index(drop=True).equals(reindexed[EDGE_COLS]), \
    "original rows were modified"
assert node_table(final).equals(node_table(reindexed)), \
    "node attributes changed"

print(f"all checks passed: {len(final):,} edges, {n_nodes:,} nodes, "
      f"{len(added)} added edges recoverable as the final {len(added)} rows")

In [ ]:
# Optional: byte-level agreement with the released files, if they are present
# under a different name. Row order is canonicalised here, so compare as multisets.
RELEASED = DATA_DIR / "released"
if RELEASED.is_dir():
    for produced, name in [(final, "Legislators_data_processed.csv"),
                           (added, "added_edges.csv")]:
        path = RELEASED / name
        if not path.exists():
            continue
        ref = pd.read_csv(path)
        cols = list(ref.columns)
        key = lambda d: d[cols].round(9).sort_values(cols).reset_index(drop=True)
        print(f"{name}: {'identical' if key(produced).equals(key(ref)) else 'DIFFERS'}")
else:
    print(f"no {RELEASED.name}/ directory - skipping comparison with released files")

## Notes

* **What Step 2 assumes.** That an observed absence of outgoing links reflects
  incomplete observation rather than genuine one-way behaviour. Where that assumption
  is untenable, the baseline to report alongside is the source data.
* **What it leaves untouched.** All 121 added edges are same-party (55 D-D, 66 R-R)
  and carry party-pair medians, so cross-party edge counts and the weight marginals
  are unchanged. Reachability, SCC structure, and the spectrum of `L_out` do change,
  by design.
* **Sensitivity knobs.** `PARTY_ORDER` (block layout), `TT`/`UT` (admissibility
  filter), the anchor rule (largest SCC), the representative rule (`members[0]`), and
  the target rule (same party, nearest belief) are the five choices worth varying in a
  robustness appendix.